## Limpieza y EDA Fbref

Voy a tomar la ruta \proyecto-integrador-laliga\data\fbref\processed y empezar un eda rapido

In [3]:
import pandas as pd
import numpy as np
import os 
from IPython.display import display # Esta es una libreria dedicada a jupiter la cual nos permite ver nuestras tablas con mejor formato

In [4]:
# Configuro Pandas para que no oculte filas, columnas o el ancho de las columnas al mostrar los DataFrames
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [5]:
# Defino la ruta para exportar mis archivos

ruta_base = "../data/fbref/processed"

# Exporto los CSVs ya procesados para su eda

partidos = pd.read_csv(os.path.join(ruta_base, "partidos_historico_completo.csv"))
tablas = pd.read_csv(os.path.join(ruta_base, "tabla_general_historico_completo.csv"))
jugadores = pd.read_csv(os.path.join(ruta_base, "jugadores_historico_completo.csv"))
equipos = pd.read_csv(os.path.join(ruta_base, "equipos_historico_completo.csv"))


In [6]:
# EDA inicial por cada tabla

print("Partidos")
print(partidos.info())

print("\nTablas")
print(tablas.info())

print("\nJugadores")
print(jugadores.info())

print("\nEquipos")
print(equipos.info())



Partidos
<class 'pandas.DataFrame'>
RangeIndex: 7980 entries, 0 to 7979
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Temporada     7980 non-null   int64  
 1   Wk            7980 non-null   float64
 2   Day           7980 non-null   str    
 3   Date          7980 non-null   str    
 4   Time          4702 non-null   str    
 5   Home          7980 non-null   str    
 6   Score         7720 non-null   str    
 7   Away          7980 non-null   str    
 8   Attendance    4191 non-null   float64
 9   Venue         7980 non-null   str    
 10  Referee       7720 non-null   str    
 11  Match Report  7980 non-null   str    
 12  Notes         1 non-null      str    
dtypes: float64(2), int64(1), str(10)
memory usage: 1.5 MB
None

Tablas
<class 'pandas.DataFrame'>
RangeIndex: 421 entries, 0 to 420
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  

In [7]:
# Veo cada tabla por separado para ver si hay valores nulos y su distribución

print("\nPartidos")
display(partidos.head())
print(partidos.describe(include='all'))
print(partidos.isnull().sum())



Partidos


,Temporada,Wk,Day,Date,Time,Home,Score,Away,Attendance,Venue,Referee,Match Report,Notes
0,2006,1.0,Sat,2006-08-26,NaN,Valencia,2–1,Real Betis,NaN,Mestalla,Muñiz Fernández,Match Report,NaN
1,2006,1.0,Sun,2006-08-27,NaN,Athletic Club,1–1,Real Sociedad,NaN,San Mamés,Undiano Mallenco,Match Report,NaN
2,2006,1.0,Sun,2006-08-27,NaN,Dep. La Coruña,3–2,Zaragoza,NaN,Nuevo Riazor,Pérez Burrull,Match Report,NaN
3,2006,1.0,Sun,2006-08-27,NaN,Recreativo,1–1,Mallorca,NaN,Nuevo Colombino,Pino Zamorano,Match Report,NaN
4,2006,1.0,Sun,2006-08-27,NaN,Osasuna,0–2,Getafe,NaN,Reyno de Navarra,Pérez Lima,Match Report,NaN


         Temporada           Wk   Day        Date           Time      Home  \
count   7980.00000  7980.000000  7980        7980           4702      7980   
unique         NaN          NaN     7        2480             67        42   
top            NaN          NaN   Sun  2007-06-17  21:00 (14:00)  Valencia   
freq           NaN          NaN  3707          10            766       399   
mean    2016.00000    19.500000   NaN         NaN            NaN       NaN   
std        6.05568    10.966543   NaN         NaN            NaN       NaN   
min     2006.00000     1.000000   NaN         NaN            NaN       NaN   
25%     2011.00000    10.000000   NaN         NaN            NaN       NaN   
50%     2016.00000    19.500000   NaN         NaN            NaN       NaN   
75%     2021.00000    29.000000   NaN         NaN            NaN       NaN   
max     2026.00000    38.000000   NaN         NaN            NaN       NaN   

       Score             Away    Attendance      Venue    Refer

In [8]:
# Filtro las  filas inválidas
# Nos aseguramos de que haya un 'Score' válido
df_partidos = partidos.dropna(subset=['Score']).copy()

# Extraigo Goles Local y Visitantes del string 'Score'
df_partidos[['Goles_Local', 'Goles_Visita']] = df_partidos['Score'].str.extract(r'(\d+)[^\d]+(\d+)')

# Convierto a valores numéricos
df_partidos['Goles_Local'] = pd.to_numeric(df_partidos['Goles_Local'], errors='coerce') # Coerce convierte los valores no convertibles a NaN
df_partidos['Goles_Visita'] = pd.to_numeric(df_partidos['Goles_Visita'], errors='coerce')

# Derivo la variable objetivo: Puntos obtenidos
condiciones = [
    df_partidos['Goles_Local'] > df_partidos['Goles_Visita'],  # Gana Local
    df_partidos['Goles_Local'] == df_partidos['Goles_Visita'], # Empate
    df_partidos['Goles_Local'] < df_partidos['Goles_Visita']   # Pierde Local
]

# El Local obtiene 3, 1 o 0
df_partidos['Puntos_Local'] = np.select(condiciones, [3, 1, 0], default=np.nan)

# El Visitante obtiene lo contrario: 0, 1 o 3
df_partidos['Puntos_Visita'] = np.where(df_partidos['Puntos_Local'] == 3, 0, 
                               np.where(df_partidos['Puntos_Local'] == 1, 1, 3))

# Formateo la fecha a datetime
df_partidos['Date'] = pd.to_datetime(df_partidos['Date'], errors='coerce')

# Limpieza de columnas
# Wk, Day, Time, Attendance, Venue, Referee, Match Report, Notes, y Score, las primeras no nos sirven score la separamos de forma diferente
columnas_analiticas = [
    'Temporada', 'Date', 'Home', 'Away', 
    'Goles_Local', 'Goles_Visita', 'Puntos_Local', 'Puntos_Visita'
]

df_partidos_limpio = df_partidos[columnas_analiticas].copy()

# Muestro resultados
print(f"Dimensiones finales de la tabla: {df_partidos_limpio.shape[0]} filas x {df_partidos_limpio.shape[1]} columnas")
display(df_partidos_limpio.head())

Dimensiones finales de la tabla: 7720 filas x 8 columnas


,Temporada,Date,Home,Away,Goles_Local,Goles_Visita,Puntos_Local,Puntos_Visita
0,2006,2006-08-26,Valencia,Real Betis,2,1,3.0,0
1,2006,2006-08-27,Athletic Club,Real Sociedad,1,1,1.0,1
2,2006,2006-08-27,Dep. La Coruña,Zaragoza,3,2,3.0,0
3,2006,2006-08-27,Recreativo,Mallorca,1,1,1.0,1
4,2006,2006-08-27,Osasuna,Getafe,0,2,0.0,3


In [9]:
# Reviso la info del df

df_partidos_limpio.info()

<class 'pandas.DataFrame'>
Index: 7720 entries, 0 to 7751
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Temporada      7720 non-null   int64         
 1   Date           7720 non-null   datetime64[us]
 2   Home           7720 non-null   str           
 3   Away           7720 non-null   str           
 4   Goles_Local    7720 non-null   int64         
 5   Goles_Visita   7720 non-null   int64         
 6   Puntos_Local   7720 non-null   float64       
 7   Puntos_Visita  7720 non-null   int64         
dtypes: datetime64[us](1), float64(1), int64(4), str(2)
memory usage: 687.2 KB


In [10]:
# Convierto la columna puntos local y visitante a int para que no haya problemas al exportar a csv
df_partidos_limpio['Puntos_Local'] = df_partidos_limpio['Puntos_Local'].astype(int)
df_partidos_limpio['Puntos_Visita'] = df_partidos_limpio['Puntos_Visita'].astype(int)

In [11]:
print("\nTabla general")
display(tablas.head())
print(tablas.describe(include='all'))
print(tablas.isnull().sum())



Tabla general


,Temporada,Rk,Squad,MP,W,D,L,GF,GA,GD,Pts,Pts/MP,Attendance,Top Team Scorer,Goalkeeper,Notes,Last 5
0,607,1,Real Madrid,38.0,23.0,7.0,8.0,66.0,40.0,26.0,76.0,2.00,NaN,Ruud van Nistelrooy - 25,Iker Casillas,→ Champions League via league finish,NaN
1,607,2,Barcelona,38.0,22.0,10.0,6.0,78.0,33.0,45.0,76.0,2.00,NaN,Ronaldinho - 21,Víctor Valdés,→ Champions League via league finish,NaN
2,607,3,Sevilla,38.0,21.0,8.0,9.0,64.0,35.0,29.0,71.0,1.87,NaN,Frédéric Kanouté - 21,Andrés Palop,→ Champions League via league finish,NaN
3,607,4,Valencia,38.0,20.0,6.0,12.0,57.0,42.0,15.0,66.0,1.74,NaN,David Villa - 16,Santiago Cañizares,→ Champions League via league finish,NaN
4,607,5,Villarreal,38.0,18.0,8.0,12.0,48.0,44.0,4.0,62.0,1.63,NaN,Diego Forlán - 19,Sebastián Viera,→ UEFA Cup via league finish,NaN


          Temporada   Rk        Squad          MP           W           D  \
count    421.000000  421          420  420.000000  420.000000  420.000000   
unique          NaN   21           42         NaN         NaN         NaN   
top             NaN    1  Real Madrid         NaN         NaN         NaN   
freq            NaN   21           21         NaN         NaN         NaN   
mean    1618.199525  NaN          NaN   36.757143   13.816667    9.123810   
std      612.080409  NaN          NaN    5.565241    6.294841    3.220446   
min      607.000000  NaN          NaN   11.000000    0.000000    0.000000   
25%     1112.000000  NaN          NaN   38.000000   10.000000    7.000000   
50%     1617.000000  NaN          NaN   38.000000   12.500000    9.000000   
75%     2122.000000  NaN          NaN   38.000000   17.000000   11.000000   
max     2627.000000  NaN          NaN   38.000000   32.000000   18.000000   

                 L          GF          GA          GD         Pts  \
count

In [12]:
# Corrijo el formato de temporada, basicamente formato temporata 06 07 07 08 y asi 
df_tabla = tablas.copy()  
df_tabla['Temporada'] = df_tabla['Temporada'].astype(str).str.zfill(4).str[:2].apply(lambda x: f"20{x}") # Lo que esto hace es tomar los dos primeros digitos y anteponerle 20, por ejemplo 06 -> 2006, 07 -> 2007, etc.
df_tabla['Temporada'] = df_tabla['Temporada'].astype(int)

# Purgo las columnas que no aportan valor predictivo
columnas_basura_tabla = ['Attendance', 'Top Team Scorer', 'Goalkeeper', 'Notes', 'Last 5']
df_tabla_limpia = df_tabla.drop(columns=[col for col in columnas_basura_tabla if col in df_tabla.columns])

# Aseguro que las estadísticas son numéricas y no tienen nulos extraños
# Las columnas como Pts, W, D, L, GF, GA, GD seguramente vengan como float, las paso a int para mayor limpieza
cols_a_entero = ['Rk', 'MP', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts']

# Primero eliminamos cualquier fila que por error de FBref no tenga un equipo válido
#df_tabla_limpia = df_tabla_limpia.dropna(subset=['Squad'])

# for col in cols_a_entero:
#   df_tabla_limpia[col] = pd.to_numeric(df_tabla_limpia[col], errors='coerce').fillna(0).astype(int)

# Ver resultados
print(f"Dimensiones finales de la tabla: {df_tabla_limpia.shape[0]} filas x {df_tabla_limpia.shape[1]} columnas")
display(df_tabla_limpia.head())

Dimensiones finales de la tabla: 421 filas x 12 columnas


,Temporada,Rk,Squad,MP,W,D,L,GF,GA,GD,Pts,Pts/MP
0,2006,1,Real Madrid,38.0,23.0,7.0,8.0,66.0,40.0,26.0,76.0,2.00
1,2006,2,Barcelona,38.0,22.0,10.0,6.0,78.0,33.0,45.0,76.0,2.00
2,2006,3,Sevilla,38.0,21.0,8.0,9.0,64.0,35.0,29.0,71.0,1.87
3,2006,4,Valencia,38.0,20.0,6.0,12.0,57.0,42.0,15.0,66.0,1.74
4,2006,5,Villarreal,38.0,18.0,8.0,12.0,48.0,44.0,4.0,62.0,1.63


In [13]:
# veo info y nulos de la tabla limpia
df_tabla_limpia.info()
df_tabla_limpia.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 421 entries, 0 to 420
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Temporada  421 non-null    int64  
 1   Rk         421 non-null    str    
 2   Squad      420 non-null    str    
 3   MP         420 non-null    float64
 4   W          420 non-null    float64
 5   D          420 non-null    float64
 6   L          420 non-null    float64
 7   GF         420 non-null    float64
 8   GA         420 non-null    float64
 9   GD         420 non-null    float64
 10  Pts        420 non-null    float64
 11  Pts/MP     420 non-null    float64
dtypes: float64(9), int64(1), str(2)
memory usage: 44.2 KB


Temporada    0
Rk           0
Squad        1
MP           1
W            1
D            1
L            1
GF           1
GA           1
GD           1
Pts          1
Pts/MP       1
dtype: int64

In [14]:
# Revisar la fila cuyo equipo está vacío
df_tabla_limpia[df_tabla_limpia["Squad"].isnull()]

# Revisar todos los equipos de la temporada 2021
df_tabla_limpia.loc[
    df_tabla_limpia["Temporada"].eq(2021),
    ["Temporada", "Rk", "Squad"]
].sort_values("Rk")

,Temporada,Rk,Squad
300,2021,1,Real Madrid
309,2021,10,Osasuna
310,2021,11,Celta Vigo
311,2021,12,Rayo Vallecano
312,2021,13,Elche
313,2021,14,Espanyol
314,2021,15,Getafe
315,2021,16,Mallorca
316,2021,17,Cádiz
317,2021,18,Granada


In [17]:
# Al ver que esto es un problema de FBref, vamos a eliminar las filas que tengan Squad nulo
df_tabla_limpia = df_tabla_limpia.dropna(subset=['Squad'])

In [15]:
print("\n Jugadores")
display(jugadores.head())
print(jugadores.describe(include='all'))
print(jugadores.isnull().sum())


 Jugadores


,Temporada,Rk,Player,Nation,Pos,Squad,Age,Born,MP,Starts,Min,90s,Gls,Ast,G+A,G-PK,PK,PKatt,CrdY,CrdR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK,Matches,-9999
0,2006,1,Dani Abalo,es ESP,FWMF,Celta Vigo,18,1987.0,1,0,13,0.1,0,NaN,NaN,0,0,NaN,0,0,0.00,NaN,NaN,0.00,NaN,Matches,d1b04adb
1,2006,2,Pato Abbondanzieri,ar ARG,GK,Getafe,33,1972.0,36,36,3215,35.7,0,0.0,0.0,0,0,0.0,3,0,0.00,0.00,0.00,0.00,0.00,Matches,ea4c5dbd
2,2006,3,Fernando de Abreu Ferreira,br BRA,DF,Racing Sant,21,1984.0,1,0,6,0.1,0,NaN,NaN,0,0,NaN,0,0,0.00,NaN,NaN,0.00,NaN,Matches,4d387d87
3,2006,4,Adriano,br BRA,DFMF,Sevilla,21,1984.0,26,25,1819,20.2,2,1.0,3.0,2,0,0.0,2,0,0.10,0.05,0.15,0.10,0.15,Matches,9d8586c4
4,2006,5,Aritz Aduriz,es ESP,FW,Athletic Club,25,1981.0,34,27,2399,26.7,9,3.0,12.0,7,2,2.0,5,3,0.34,0.11,0.45,0.26,0.38,Matches,293211e1


           Temporada            Rk    Player  Nation    Pos    Squad    Age  \
count   11765.000000  11765.000000     11765   11760  11764    11765  11762   
unique           NaN           NaN      3590     114     10       42    520   
top              NaN           NaN  Juanfran  es ESP     MF  Sevilla     26   
freq             NaN           NaN        30    7060   3331      628    875   
mean     2016.182150    281.528602       NaN     NaN    NaN      NaN    NaN   
std         6.030938    163.307463       NaN     NaN    NaN      NaN    NaN   
min      2006.000000      1.000000       NaN     NaN    NaN      NaN    NaN   
25%      2011.000000    141.000000       NaN     NaN    NaN      NaN    NaN   
50%      2016.000000    281.000000       NaN     NaN    NaN      NaN    NaN   
75%      2021.000000    421.000000       NaN     NaN    NaN      NaN    NaN   
max      2026.000000    617.000000       NaN     NaN    NaN      NaN    NaN   

                Born            MP        Starts   

In [19]:
# Purgo columnas que no aportan valor predictivo y la posición (para optimizar el cálculo de distancias en ML)
columnas_basura_jug = ['Matches', '-9999', 'Pos']
df_jugadores_limpia = jugadores.drop(columns=[col for col in columnas_basura_jug if col in jugadores.columns])

# Limpieza de variables demograficas
# Quitar el prefijo de nacionalidad
if 'Nation' in df_jugadores_limpia.columns:
    df_jugadores_limpia['Nation'] = df_jugadores_limpia['Nation'].astype(str).str.split(' ').str[-1].replace('nan', 'Desconocido')

# Quito los días de la edad para volverla un número entero (ej. '24-162' -> 24)
if 'Age' in df_jugadores_limpia.columns:
    df_jugadores_limpia['Age'] = df_jugadores_limpia['Age'].astype(str).str.split('-').str[0]
    df_jugadores_limpia['Age'] = pd.to_numeric(df_jugadores_limpia['Age'], errors='coerce')

# Relleno vacíos
# Si FBref dejó un NaN en stats como Asistencias o Penales, matemáticamente significa un 0
cols_numericas = df_jugadores_limpia.select_dtypes(include=['float64', 'int64']).columns
df_jugadores_limpia[cols_numericas] = df_jugadores_limpia[cols_numericas].fillna(0)

# Convierto la edad a entero
df_jugadores_limpia['Age'] = df_jugadores_limpia['Age'].astype(int)

# Veo resultados
print(f"Dimensiones finales de la tabla: {df_jugadores_limpia.shape[0]} filas x {df_jugadores_limpia.shape[1]} columnas")
display(df_jugadores_limpia.head())

Dimensiones finales de la tabla: 11765 filas x 24 columnas


,Temporada,Rk,Player,Nation,Squad,Age,Born,MP,Starts,Min,90s,Gls,Ast,G+A,G-PK,PK,PKatt,CrdY,CrdR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK
0,2006,1,Dani Abalo,ESP,Celta Vigo,18,1987.0,1,0,13,0.1,0,0.0,0.0,0,0,0.0,0,0,0.00,0.00,0.00,0.00,0.00
1,2006,2,Pato Abbondanzieri,ARG,Getafe,33,1972.0,36,36,3215,35.7,0,0.0,0.0,0,0,0.0,3,0,0.00,0.00,0.00,0.00,0.00
2,2006,3,Fernando de Abreu Ferreira,BRA,Racing Sant,21,1984.0,1,0,6,0.1,0,0.0,0.0,0,0,0.0,0,0,0.00,0.00,0.00,0.00,0.00
3,2006,4,Adriano,BRA,Sevilla,21,1984.0,26,25,1819,20.2,2,1.0,3.0,2,0,0.0,2,0,0.10,0.05,0.15,0.10,0.15
4,2006,5,Aritz Aduriz,ESP,Athletic Club,25,1981.0,34,27,2399,26.7,9,3.0,12.0,7,2,2.0,5,3,0.34,0.11,0.45,0.26,0.38


In [16]:
print("\nEquipos")
display(equipos.head())
print(equipos.describe(include='all'))
print(equipos.isnull().sum())


Equipos


,Temporada,Squad,# Pl,Age,Poss,MP,Starts,Min,90s,Gls,Ast,G+A,G-PK,PK,PKatt,CrdY,CrdR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK
0,2006,Athletic Club,26,25.7,NaN,38,418,3420,38.0,42,27,69,36,6,7,93,7,1.11,0.71,1.82,0.95,1.66
1,2006,Atlético Madrid,26,25.4,NaN,38,418,3420,38.0,44,33,77,38,6,10,118,10,1.16,0.87,2.03,1.00,1.87
2,2006,Barcelona,22,27.3,NaN,38,418,3420,38.0,76,45,121,69,7,8,85,7,2.00,1.18,3.18,1.82,3.00
3,2006,Celta Vigo,26,26.9,NaN,38,418,3420,38.0,40,27,67,33,7,8,113,11,1.05,0.71,1.76,0.87,1.58
4,2006,Dep. La Coruña,27,25.6,NaN,38,418,3420,38.0,31,18,49,28,3,4,121,11,0.82,0.47,1.29,0.74,1.21


          Temporada          Squad        # Pl         Age        Poss  \
count    420.000000            420  420.000000  420.000000  260.000000   
unique          NaN             42         NaN         NaN         NaN   
top             NaN  Athletic Club         NaN         NaN         NaN   
freq            NaN             21         NaN         NaN         NaN   
mean    2016.000000            NaN   28.011905   26.887619   50.015385   
std        6.062522            NaN    3.305070    1.085152    6.295386   
min     2006.000000            NaN   22.000000   23.800000   34.800000   
25%     2011.000000            NaN   25.000000   26.200000   45.675000   
50%     2016.000000            NaN   28.000000   26.900000   48.800000   
75%     2021.000000            NaN   30.000000   27.700000   53.425000   
max     2026.000000            NaN   40.000000   30.400000   69.600000   

                MP      Starts          Min         90s         Gls  \
count   420.000000  420.000000   420.000

In [21]:
# Veo los nulos de la tabla de equipos
equipos.info()
equipos.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 22 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Temporada  420 non-null    int64  
 1   Squad      420 non-null    str    
 2   # Pl       420 non-null    int64  
 3   Age        420 non-null    float64
 4   Poss       260 non-null    float64
 5   MP         420 non-null    int64  
 6   Starts     420 non-null    int64  
 7   Min        420 non-null    int64  
 8   90s        420 non-null    float64
 9   Gls        420 non-null    int64  
 10  Ast        420 non-null    int64  
 11  G+A        420 non-null    int64  
 12  G-PK       420 non-null    int64  
 13  PK         420 non-null    int64  
 14  PKatt      420 non-null    int64  
 15  CrdY       420 non-null    int64  
 16  CrdR       420 non-null    int64  
 17  Gls.1      420 non-null    float64
 18  Ast.1      420 non-null    float64
 19  G+A.1      420 non-null    float64
 20  G-PK.1     420 non-nu

Temporada      0
Squad          0
# Pl           0
Age            0
Poss         160
MP             0
Starts         0
Min            0
90s            0
Gls            0
Ast            0
G+A            0
G-PK           0
PK             0
PKatt          0
CrdY           0
CrdR           0
Gls.1          0
Ast.1          0
G+A.1          0
G-PK.1         0
G+A-PK         0
dtype: int64

In [ ]:
# Reviso cuales son las columnas que tienen nulos y si son relevantes para el análisis
df_equipos_nulos = equipos.isnull().sum()
print(df_equipos_nulos[df_equipos_nulos > 0])

Poss    160
dtype: int64


['Poss']

In [ ]:
# Poss es el porcentaje de posesión del equipo. Para no introducir sesgo
# en los registros históricos sin este dato, usamos el valor neutral de 50%.
df_equipos_limpia = equipos.copy()

# 1. Imputar los nulos históricos de posesión con el promedio neutral de la liga
if 'Poss' in df_equipos_limpia.columns:
    df_equipos_limpia['Poss'] = df_equipos_limpia['Poss'].fillna(50.0)

# Debo renombras las columnas ".1" que genera FBref
# FBref nombra "Gls.1" a los Goles por cada 90 minutos. Esto es confuso para el EDA y ML.
renombres = {
    'Gls.1': 'Gls_per90',
    'Ast.1': 'Ast_per90',
    'G+A.1': 'G+A_per90',
    'G-PK.1': 'G-PK_per90',
    'G+A-PK': 'G+A-PK_per90'
}
df_equipos_limpia = df_equipos_limpia.rename(columns=renombres)

# Me aseguro que las columnas numéricas no tengan nulos, si los hay los relleno con 0
cols_numericas = df_equipos_limpia.select_dtypes(include='number').columns
df_equipos_limpia[cols_numericas] = df_equipos_limpia[cols_numericas].fillna(0)

# Ver resultados
print(f"Dimensiones finales de la tabla: {df_equipos_limpia.shape[0]} filas x {df_equipos_limpia.shape[1]} columnas")
print("\nMuestra de las columnas renombradas y posesión imputada:")
display(df_equipos_limpia[['Temporada', 'Squad', 'Poss', 'Gls', 'Gls_per90']].head())

Dimensiones finales de la tabla: 420 filas x 22 columnas

Muestra de las columnas renombradas y posesión imputada:


,Temporada,Squad,Poss,Gls,Gls_per90
0,2006,Athletic Club,50.0,42,1.11
1,2006,Atlético Madrid,50.0,44,1.16
2,2006,Barcelona,50.0,76,2.00
3,2006,Celta Vigo,50.0,40,1.05
4,2006,Dep. La Coruña,50.0,31,0.82


In [26]:
# Exporto los CSVs limpios para su posterior análisis y modelado
df_partidos_limpio.to_csv(os.path.join(ruta_base, "partidos_historico_limpio.csv"), index=False)
df_tabla_limpia.to_csv(os.path.join(ruta_base, "tabla_general_historico_limpio.csv"), index=False)
df_jugadores_limpia.to_csv(os.path.join(ruta_base, "jugadores_historico_limpio.csv"), index=False)
df_equipos_limpia.to_csv(os.path.join(ruta_base, "equipos_historico_limpio.csv"), index=False)